In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

#データを読み解く
import pandas as pd 
train=pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test=pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")
print(train.head(5))
print(test.head(5))

print(train.shape)
print(test.shape)
print(train.info())
print(test.info())

#欠損データの特定
print(train.isnull().sum())
miss_ration_train=(train.isnull().sum()/len(train))*100
print("訓練データの欠損率:",miss_ration_train)

print(test.isnull().sum())
miss_ration_test=(test.isnull().sum()/len(test))*100
print("テストデータの欠損率:",miss_ration_test)

#Transportedの処理の判断
print(train["Transported"].value_counts())

#HomePlanetとTransportedの関係
train.groupby("HomePlanet")["Transported"].mean()

#CryoSleepとTransportedの関係
train.groupby("CryoSleep")["Transported"].mean()

#AgeとTransportedの関係
bins=[0,10,20,30,40,50,60,70,80,200]
labels=["10歳未満","10代","20代","30代","40代","50代","60代","70代","80歳以上"]
population_distribution=pd.cut(train["Age"],bins=bins,labels=labels,right=False)
print(population_distribution.value_counts())
train.groupby(population_distribution)["Transported"].mean()

#RoomServiceとTransportedの関係
print(train.groupby("Transported")["RoomService"].mean())

#FoodCourtとTransportedの関係
print(train.groupby("Transported")["FoodCourt"].mean())

#ShoppingMallとTransportedの関係
print(train.groupby("Transported")["ShoppingMall"].mean())

#SpaとTransportedの関係
print(train.groupby("Transported")["Spa"].mean())

#VRDeckとTransportedの関係
print(train.groupby("Transported")["VRDeck"].mean())

#Cabinの分析
print(train["Cabin"].head(5))
train[["Deck","Num","side"]]=train["Cabin"].str.split("/",expand=True)
test[["Deck","Num","side"]]=test["Cabin"].str.split("/",expand=True)
print(train[["Deck","Num","side"]])
print("Desk:",train["Deck"].dtype)
print("Num:",train["Num"].dtype)
print("side:",train["side"].dtype)
train["Num"]=pd.to_numeric(train["Num"])
print(train.groupby("Deck")["Transported"].mean())
print(train.groupby("side")["Transported"].mean())
print(train["Deck"].isnull().sum())
print(train["Num"].isnull().sum())
print(train["side"].isnull().sum())

#Cabinの欠損値の補完
mode_Deck=train["Deck"].mode()[0]
Num_median=train["Num"].median()
mode_side=train["side"].mode()[0]
print("Deckの最頻値:",mode_Deck)
print("Numの中央値:",Num_median)
print("sideの最頻値:",mode_side)

train["Deck"]=train["Deck"].fillna(mode_Deck)
train["Num"]=train["Num"].fillna(Num_median)
train["side"]=train["side"].fillna(mode_side)

test["Deck"]=test["Deck"].fillna(mode_Deck)
test["Num"]=test["Num"].fillna(Num_median)
test["side"]=test["side"].fillna(mode_side)

data_Num=pd.to_numeric(train["Num"])
data_Deck=pd.get_dummies(train["Deck"])
data_side=pd.get_dummies(train["side"])
print("Deck:",data_Deck.head(5))
print("side:",data_side.head(5))

data_Num_test=pd.to_numeric(test["Num"])
data_Deck_test=pd.get_dummies(test["Deck"])
data_side_test=pd.get_dummies(test["side"])
print("Deck:",data_Deck_test.head(5))
print("side:",data_side_test.head(5))

#Cabinのデータ修正
new_train=train.drop(["Num","Deck","side"],axis=1)
new_test=test.drop(["Num","Deck","side"],axis=1)

train1=pd.concat([new_train,data_Deck,data_side,data_Num],axis=1)
test1=pd.concat([new_test,data_Deck_test,data_side_test,data_Num_test],axis=1)

print(train1.shape)
print(test1.shape)
print(train1.columns)
print(test1.columns)

train2=train1.drop(["Cabin"],axis=1)
test2=test1.drop(["Cabin"],axis=1)

print(train2.dtypes)
print(test2.dtypes)

#PassengerIdとNameとTransportedを削除して不要な列を整理
train3=train2.drop(["PassengerId","Name"],axis=1)
test3=test2.drop(["PassengerId","Name"],axis=1)

print("訓練用:",train3.dtypes)
print("テスト用:",test3.dtypes)

train4=train3.drop(["Transported",],axis=1)
data_train4_test3 = pd.concat([train4, test3], axis=0)
data_train4_test3_bit=pd.get_dummies(data_train4_test3)

print("結合結果:", data_train4_test3.shape)
print("列名:", data_train4_test3.columns.tolist())
print(data_train4_test3_bit.shape)
print(data_train4_test3_bit.dtypes)

#trainとtestに分割
train5=data_train4_test3_bit.iloc[:8693]
test4=data_train4_test3_bit.iloc[8693:12970]
print(train5.shape)
print(test4.shape)

#変数の設定
x=train5
y=train["Transported"]
print(y.shape)
print(y.value_counts())

#データの分割
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

#ランダムフォレスト
from sklearn.ensemble import RandomForestClassifier
model=RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
#学習と予測
model.fit(x_train,y_train)
y_pred=model.predict(x_test)

#正確率
from sklearn.metrics import accuracy_score
accuracy=accuracy_score(y_test, y_pred)
print("正確率:",accuracy)

#特徴量重要度
import numpy as np
importance=model.feature_importances_
print(importance.dtype)
print(importance[:5])

#DataFrameの作成
Data=pd.DataFrame({
    '特徴量名':x_train.columns,
    '特徴量重要度':model.feature_importances_
})
print(Data.sort_values(by="特徴量重要度",ascending=False))

#予測精度の確認
train_pred=model.predict(x_train)
test_pred=model.predict(x_test)

print("Train Accuracy:",accuracy_score(y_train, train_pred))
print("Test Accuracy:",accuracy_score(y_test, test_pred))

#最も良いパラメーターを探す
from sklearn.model_selection import GridSearchCV
model1=RandomForestClassifier(random_state=42)
param_grid={
    "n_estimators":[1,10,20,30,40,50,100],
    "max_depth":[1,10,20,30,40,None]
}
grid=GridSearchCV(
    model1,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring="accuracy"
)
grid.fit(x_train,y_train)
print("最も良いモデル:",grid.best_estimator_)
print("最も良いパラメーター:",grid.best_params_)
print("最も良い正確性:", grid.best_score_)

#最も良いモデルの予測精度
best_model=grid.best_estimator_
best_pred=best_model.predict(x_test)

print("最も良いモデルの正確性:", accuracy_score(y_test, best_pred))

train_pre_best=best_model.predict(x_train)
print("訓練データの正確性:", accuracy_score(y_train,train_pre_best))

#特徴量を変化
x_train_new=x_train.drop(["VIP_False","VIP_True"],axis=1)
x_test_new=x_test.drop(["VIP_False","VIP_True"],axis=1)
print(x_train_new.shape)
print(x_test_new.shape)
best_model.fit(x_train_new,y_train)
y_pred_new=best_model.predict(x_test_new)
print("特徴量を変化させた時の正確性:",accuracy_score(y_test,y_pred_new))

#最終データ
import pandas as pd
model.fit(x_train,y_train)
test_pred_last=model.predict(test4)
print(test_pred_last[:5])

#提出物
submission=pd.DataFrame({
    'PassengerId':test["PassengerId"],
    'Transported':test_pred_last
})
print(submission[:5])
submission.to_csv("submission.csv",index=False)

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv
  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4      

/tmp/ipykernel_16/2783637827.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  train.groupby(population_distribution)["Transported"].mean()


正確率: 0.7958596894767107
float64
[0.132365   0.08296779 0.07858609 0.06755167 0.0905137 ]
                         特徴量名    特徴量重要度
16                        Num  0.168080
0                         Age  0.132365
4                         Spa  0.090514
1                 RoomService  0.082968
5                      VRDeck  0.082471
2                   FoodCourt  0.078586
21             CryoSleep_True  0.076338
20            CryoSleep_False  0.069660
3                ShoppingMall  0.067552
17           HomePlanet_Earth  0.021738
18          HomePlanet_Europa  0.013996
12                          G  0.012614
10                          E  0.011897
11                          F  0.011793
14                          P  0.011400
24    Destination_TRAPPIST-1e  0.011102
15                          S  0.010480
19            HomePlanet_Mars  0.009516
22    Destination_55 Cancri e  0.008905
23  Destination_PSO J318.5-22  0.006047
8                           C  0.005606
25                  VIP_False  

10歳未満のTransported率が約71%と高い。
220〜30代は約46〜47%と比較的低い。
40代以上はおおむね43〜51%の範囲にある。

Europa出身者は約65.9%と最も高い。
Earth出身者は約42.4%と最も低い。
Mars出身者は約52.3%

Spa：Falseの平均支出額がTrueの約9倍。
VRDeck：Falseの平均支出額がTrueの約8倍。
RoomService：Falseの平均支出額がTrueの約6倍。

Deckごとの輸送率はB・Cの輸送率が高く、E・Tは低い。sideごとの輸送率はsの方が10%程度大きい。

importanceよりどこの席にいるかによって避難にかかる時間が変化するので、numが最も重要になったと考察できる。また、子供は優先的に避難させようとする心理状況から10歳以下や10代のTransported率が高くなっていたので、重要性が高くなっていた。更に、vipだろうがvipでなかろうが避難誘導が変わらなかったためvipの重要性が全然高くなかったと考察できる。

初期モデル訓練用データの予測データの正確性はほぼ100%で極めて精度が高いのに、テストデータの予測の正確性は80%程度で大きな差があり過学習になっている。

初期設定ではランダムフォレストの設定が深く木の本数も多く、木の深さに制限がなかったが、訓練データの精度が極めて高くそれに伴い過学習が大きく起こってしまっていた。しかし、最適モデルではランダムフォレストの設定をもう少し単純にしてまた木の深さに制限を設けたので訓練データの精度が下がってしまったが過学習が抑制された。

最適モデルでは訓練用データの予測の正確性が87%程度で、テストデータの予測の正確性は80%程度になった。

特徴量重要度が低かったVIPのデータを削除して検証したところ、予測精度は79.59%から79.24%に低下した。このことから、VIPは重要度こそ低いものの、予測に大きな影響を与えて、モデルに必要な特徴量の一つである可能性が考えられる。